# Classical Hopfield Networks
A classical Hopfield network is a fully connected, undirected graph consisting of $N$ nodes (neurons), where each node has a binary state $s = \pm 1$. Each node is connected to every other node, but not to itself. The connection weights between nodes $i$ and $j$, denoted $w_{ij} \in \mathbf{W}$, are determined using a **Hebbian learning rule**.

> __Hebbian learning__
>
> * __Hebbian Learning Rule__: The [Hebbian learning rule](https://en.wikipedia.org/wiki/Hebbian_theory), proposed by [Donald Hebb in 1949](https://en.wikipedia.org/wiki/Donald_O._Hebb), states that synaptic connections between neurons are strengthened when they activate (fire) simultaneously, forming the biological basis for __associative learning__. This "fire together, wire together" principle underpins unsupervised learning in neural networks, linking co-active nodes to enable pattern storage and recall.
> * __Different?__ Unlike the previous examples of learning, e.g., logistic regression or any of the online learning approaches that we looked at previously, the parameters (weights) in a [Hopfield network](https://en.wikipedia.org/wiki/Hopfield_network) are entirely specified by the memories we want to encode. Thus, we do not need to search for weights or learn them by experimenting with the world. Instead, we can directly compute the weights from the memories we want to encode.
> * __Recurrent?__ A Hopfield network is a special type of [recurrent neural network](https://en.wikipedia.org/wiki/Recurrent_neural_network) in which recurrence is used to settle into a stable pattern iteratively. It is considered recurrent because its units are symmetrically and recurrently connected, allowing the network to evolve toward an energy minimum over time.
> 
> The Hebbian learning rule uses only local computations without explicit training iterations, providing a biological basis for memory encoding that operates without specialized hardware or extended optimization cycles.

<div>
    <center>
        <img src="figs/Fig-EnergyLandscape-Schematic.svg" width="580"/>
    </center>
</div>

### Encoding memories into a Hopfield network
The central idea behind Hopfield networks is that we can encode memories into the weights of the network using the Hebbian learning rule. These memories are stored in stable local minima energy states (attractors). To recover a memory, we initialize the network with a noisy or partial version of the memory and let the network dynamics evolve until it settles into the closest stored memory.

Suppose we wish our network to memorize $K$ images, where each image is an $n\times{n}$ collection of black and white pixels represented as a vector $\mathbf{s}_{i}\in\left\{-1,1\right\}^{n^2}$. We encode the image using the following rule: if the pixel is white, we set the memory value to `1`, and if the pixel is black, we set the memory value to `-1`. Then, the weights that encode these $K$ images are given by:
$$
\begin{equation*}
\mathbf{W} = \frac{1}{K}\cdot\sum_{i=1}^{K}\mathbf{s}_{i}\otimes\mathbf{s}_{i}^{\top}
\end{equation*}
$$
where $\mathbf{s}_{i}$ denotes the state (pixels) of the image we want to memorize, and $\otimes$ denotes the outer product. Thus, the weights are like an average of all of our memories!

> __How big can $K$ be?__: The maximum theoretical storage limit $K_{\text{max}}$ of a classical Hopfield network, using the standard Hebbian learning rule, is approximately $K_{max}\sim{0.138}{N}$, where $N$ is the number of neurons in the network. Thus, the network can reliably store about 14% of its size in patterns before retrieval errors become significant due to interference between stored patterns.

Suppose we've encoded $K$ images and want to retrieve one of them. This seems magical. How does it work? 



## Algorithm: Memory retrieval
Each memory in a Hopfield network is encoded as a _local minimum_ of a global energy function. Thus, during memory retrieval, when we supply a random state vector $\hat{\mathbf{s}}$, we will recover the _closest_ memory encoded in the network to where we start.
The overall energy of the network is given by:
$$
\begin{equation*}
E(\mathbf{s}) = -\frac{1}{2}\,\sum_{ij}w_{ij}s_{i}s_{j} - \sum_{i}b_{i}s_{i}
\end{equation*}
$$
where $w_{ij}\in\mathbf{W}$ are the weights of the network, and $b_{i}$ is a bias term (typically set to zero but can be used to control the activation threshold of the neurons).

Let's outline some pseudocode for the memory retrieval algorithm.

__Initialize__: Compute the weights $w_{ij}\in\mathbf{W}$ using the Hebbian learning rule. Initialize the network with a random state $\mathbf{s}$. Set $\texttt{converged}\gets\texttt{false}$, the iteration counter $t\gets{1}$, maximum iterations $\texttt{maxiter} = 10N$ (where $N$ is the number of neurons), and patience parameter $\texttt{patience}$.

> **Patience Parameter** 
> 
> The patience parameter determines how many consecutive identical states are required to declare convergence. It is a practical heuristic that balances convergence detection with computational efficiency. Classical Hopfield networks can occasionally get stuck in short oscillation cycles (e.g., alternating between a few states). Requiring a fixed number of consecutive identical states ensures the network has truly converged to a stable attractor rather than just pausing briefly or terminating prematurely. 

__Track__: Initialize a queue $\texttt{S}$ to store the last $\texttt{patience}$ state vectors.

While not $\texttt{converged}$ __do__:
1. Store the current state: $\mathbf{s}_{\text{old}} \gets \mathbf{s}$.
2. **Asynchronous update**: Choose a random node $i$ and compute a new state $s_{i}^{\prime}$ using the update rule: $s_{i}^{\prime} \leftarrow \texttt{sign}\left(\sum_{j}w_{ij}s_{j}-b_{i}\right)$, where $\texttt{sign}(\cdot)$ is the sign function and $b_{i}$ is a bias (threshold) parameter.
3. Update the network state: $\mathbf{s} \leftarrow \mathbf{s}^{\prime}$ (only neuron $i$ changes).
4. Add current state to history: $\texttt{S}\gets\texttt{S} \cup \{\mathbf{s}\}$.
5. **Check for convergence**: There are several criteria we can use to stop the iteration and determine if the network has converged:
   - **State stability**: If the state history $\texttt{S}$ contains $\texttt{patience}$ states and all states in the history are identical (Hamming distance = 0 between all consecutive pairs), then set $\texttt{converged}\gets\texttt{true}$.
   - **Memory retrieval**: Alternatively, if the current state $\mathbf{s}$ exactly matches any stored memory pattern $\mathbf{s}_k$ (Hamming distance = 0), then set $\texttt{converged}\gets\texttt{true}$.
   - **Energy minimum reached**: If the energy $E(\mathbf{s})$ equals or falls below the __true minimum__, then set $\texttt{converged}\gets\texttt{true}$.
   - __Max iterations__: If $t \geq \texttt{maxiter}$, set $\texttt{converged}\gets\texttt{true}$. Notify that maximum iterations reached without convergence.
6. If the length of the state history queue $\texttt{S}$ exceeds $\texttt{patience}$ length, remove the oldest state.
7. Update iteration counter: $t \leftarrow t + 1$.

> **Hamming Distance**: The Hamming distance between two binary vectors $\mathbf{a}$ and $\mathbf{b}$ is defined as $H(\mathbf{a}, \mathbf{b}) = \sum_{i=1}^{N} \mathbb{I}[a_i \neq b_i]$, where $\mathbb{I}[\cdot]$ is the indicator function. For convergence, we check if $H(\mathbf{s}_{\text{current}}, \mathbf{s}_{\text{previous}}) = 0$, meaning the states are identical.

### Convergence
Classical Hopfield networks have strong theoretical convergence guarantees that make them particularly appealing for associative memory tasks.

* **Guaranteed Convergence**: The asynchronous update rule ensures that the network's energy function $E(\mathbf{s})$ is monotonically non-increasing with each neuron update. Since the state space is finite (each neuron can only be in one of two states: $\pm 1$), and the energy has a lower bound, the network is **guaranteed to converge** to a stable state or a short limit cycle.
* **Energy Landscape**: Each stored memory pattern $\mathbf{s}_k$ corresponds to a local minimum in the energy landscape. When the network is initialized with a partial or noisy version of a stored pattern, the iterative updates guide the system downhill in energy space toward the nearest local minimum, effectively "cleaning up" the corrupted input and retrieving the complete memory.

The classical Hopfield network's convergence properties make it a robust model for associative memory, capable of retrieving stored patterns from incomplete or noisy inputs, provided the number of stored patterns does not exceed the network's capacity.

> **Convergence** 
>
> In practice, classical Hopfield networks typically converge quickly: in the best case, when initialized close to a stored pattern, convergence can occur in $\mathcal{O}(1)$ steps. However, convergence time varies based on network size and the number of stored patterns. In the worst case, convergence takes $\mathcal{O}(N^2)$ steps, though this is rare in practice.
>
> **Limitations**: While convergence is guaranteed, the network may converge to:
> - **Spurious attractors**: Stable states that are not stored memories but arise from interference between patterns
> - **Incomplete patterns**: Local minima that represent corrupted versions of stored memories
> - **Wrong memories**: The network may converge to a different stored pattern than intended if the initial state is equidistant from multiple memories
> - **Antipatterns**: The network may converge to the bitwise inverse of a stored memory (e.g., if $\mathbf{s}_k$ is stored, then $-\mathbf{s}_k$ is also a stable attractor). This occurs because the Hebbian learning rule creates symmetric energy wells around both a pattern and its inverse.

The convergence behavior degrades as the number of stored patterns approaches the theoretical limit of $K \approx 0.138N$, where pattern interference becomes significant and spurious attractors multiply.
___